# Federated Learning MVP with Homomorphic Encryption

This notebook runs the Federated Learning MVP in Google Colab with Streamlit.

**What this demo does:**
- Demonstrates privacy-preserving federated learning for fraud detection
- Uses homomorphic encryption (TenSEAL) for secure model aggregation
- Provides LLM-powered explanations of the process

**Prerequisites:**
- OpenAI API key (optional, for LLM explanations)
- Get one at: https://platform.openai.com/api-keys

---

## Step 1: Install Dependencies

This will install all required packages. **This may take 2-3 minutes.**

In [ ]:
%%capture
# Install all required packages
!pip install streamlit tensorflow numpy pandas matplotlib seaborn tenseal openai python-dotenv

# Install localtunnel for creating public URLs (no authentication required)
!npm install -g localtunnel

## Step 2: Clone the Repository

Clone the federated learning repository from GitHub.

In [ ]:
# Clone the repository
!git clone https://github.com/almosttomorrow/federatedlearning.git
%cd federatedlearning

## Step 3: Configure API Key (Optional)

Set your OpenAI API key to enable LLM explanations.

**Note:** The app works without an API key, but LLM explanations will be disabled.

In [ ]:
import os
from getpass import getpass

# Prompt for API key (input will be hidden)
print("Enter your OpenAI API key (or press Enter to skip):")
api_key = getpass("API Key: ")

if api_key:
    # Create .env file with the API key
    with open('.env', 'w') as f:
        f.write(f'OPENAI_API_KEY={api_key}\n')
        f.write('OPENAI_MODEL=gpt-3.5-turbo\n')
    print("✓ API key configured successfully!")
else:
    print("⚠ Skipping API key configuration. LLM explanations will be disabled.")

## Step 4: Run the Streamlit App

This will start the Streamlit app and create a public URL using **localtunnel**.

**The URL will appear below - click it to access the app!**

**Note:** 
- The app will run until you stop this cell (Runtime > Interrupt execution)
- You may need to click "Click to Continue" when opening the localtunnel URL
- If the localtunnel URL doesn't work, try the alternative method in Step 5

In [ ]:
import subprocess
import time
import threading
import re

# Kill any existing Streamlit processes
!pkill -9 streamlit

# Start Streamlit in the background
print("Starting Streamlit app...")
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for Streamlit to start
print("Waiting for Streamlit to initialize...")
time.sleep(10)

# Start localtunnel
print("Creating public URL with localtunnel...\n")
lt_process = subprocess.Popen(
    ['lt', '--port', '8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Read the URL from localtunnel output
def read_url():
    for line in iter(lt_process.stdout.readline, ''):
        if 'your url is:' in line.lower():
            url = line.split('is:')[-1].strip()
            print("\n" + "="*70)
            print("🎉 Streamlit app is running!")
            print("="*70)
            print(f"\n📱 Access your app at: {url}\n")
            print("="*70)
            print("\n⚠️  Important Notes:")
            print("   - Keep this cell running! Stop it to shut down the app.")
            print("   - You may see a warning page - click 'Click to Continue'")
            print("   - If the URL doesn't work, try the alternative method below\n")
            break

url_thread = threading.Thread(target=read_url)
url_thread.start()
url_thread.join(timeout=30)

# Keep the cell running
try:
    streamlit_process.wait()
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    streamlit_process.terminate()
    lt_process.terminate()

## Step 5: Alternative Method - Using ngrok (with free account)

If localtunnel doesn't work, use this method instead:

1. **Sign up for free ngrok account**: https://dashboard.ngrok.com/signup
2. **Get your authtoken**: https://dashboard.ngrok.com/get-started/your-authtoken
3. **Run the cell below** and paste your authtoken when prompted

In [ ]:
# Alternative method using ngrok (requires free signup)
!pip install -q pyngrok

import subprocess
from pyngrok import ngrok, conf
from getpass import getpass
import time

# Get ngrok authtoken
print("Get your free ngrok authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken\n")
authtoken = getpass("Enter your ngrok authtoken: ")

if authtoken:
    # Set the authtoken
    conf.get_default().auth_token = authtoken
    
    # Kill any existing Streamlit processes
    !pkill -9 streamlit
    
    # Start Streamlit
    print("\nStarting Streamlit app...")
    streamlit_process = subprocess.Popen(
        ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    # Wait for Streamlit to start
    time.sleep(10)
    
    # Create ngrok tunnel
    print("Creating public URL...\n")
    public_url = ngrok.connect(8501)
    
    print("="*70)
    print("🎉 Streamlit app is running!")
    print("="*70)
    print(f"\n📱 Access your app at: {public_url}\n")
    print("="*70)
    print("\n⚠️  Keep this cell running! Stop it to shut down the app.\n")
    
    # Keep the cell running
    try:
        streamlit_process.wait()
    except KeyboardInterrupt:
        print("\n🛑 Shutting down...")
        streamlit_process.terminate()
        ngrok.kill()
else:
    print("⚠️ No authtoken provided. Please get one from https://dashboard.ngrok.com/signup")

## How to Use the App

Once you click the URL above, you'll see the Federated Learning MVP interface.

### Workflow:

1. **Generate Synthetic Data** - Click to create fraud detection datasets for multiple banks
2. **Train Local Models** - Each bank trains a model on their private data
3. **Encrypt Weights** - Model weights are encrypted using homomorphic encryption
4. **Aggregate & Create Global Model** - Encrypted weights are aggregated securely
5. **Generate Explanation** - LLM explains the entire process (requires API key)

### Customization:

Use the **sidebar** to adjust:
- Number of banks (2-5)
- Samples per bank (500-2000)
- Training epochs (5-20)

---

## Stop the App

To stop the app, use **Runtime > Interrupt execution** or run the cell below.

In [ ]:
# Stop all processes
!pkill -9 streamlit
!pkill -9 node
print("✓ App stopped successfully!")

## Troubleshooting

### Common Issues:

**1. "Module not found" errors:**
- Re-run Step 1 to reinstall dependencies

**2. App won't start:**
- Run the "Stop the App" cell
- Wait 10 seconds
- Re-run Step 4 or 5

**3. Localtunnel URL not working:**
- Try the ngrok method in Step 5 instead
- Or try incognito/private browsing mode
- Click "Click to Continue" on the warning page

**4. ngrok authentication error:**
- Sign up for free at https://dashboard.ngrok.com/signup
- Get your authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
- Use Step 5 with your authtoken

**5. LLM explanations not working:**
- Verify your OpenAI API key is correct
- Check your OpenAI account has credits
- Re-run Step 3 with the correct API key

**6. Memory errors:**
- Use smaller values in the sidebar (fewer banks, fewer samples)
- Restart the runtime (Runtime > Restart runtime)

---

## Additional Resources

- **GitHub Repository:** https://github.com/almosttomorrow/federatedlearning
- **Issues:** https://github.com/almosttomorrow/federatedlearning/issues
- **OpenAI API Keys:** https://platform.openai.com/api-keys
- **ngrok Signup:** https://dashboard.ngrok.com/signup

---

**Enjoy exploring federated learning!** 🚀